<a href="https://colab.research.google.com/github/DavieK1/DAVEMUNGAIK.-/blob/main/Fetching_Weather_Data_via_Open_Meteo_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
#Practical Lab: Fetching Weather Data via Open-Meteo API

#python

import requests
import pandas as pd
from google.colab import files

# Nairobi coordinates
latitude = -1.286389
longitude = 36.817223

# Historical weather data (past 30 days only)
url = (
    f"https://api.open-meteo.com/v1/forecast?"
    f"latitude={latitude}&"
    f"longitude={longitude}&"
    f"past_days=30&"
    f"forecast_days=0&"
    f"hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
)

# Send API request
response = requests.get(url)

# Convert to JSON
data = response.json()

# Create DataFrame
weather_df = pd.DataFrame({
    "Time": data["hourly"]["time"],
    "Temperature_C": data["hourly"]["temperature_2m"],
    "Humidity_Percent": data["hourly"]["relative_humidity_2m"],
    "WindSpeed_mps": data["hourly"]["wind_speed_10m"]
})

# Convert Time column
weather_df["Time"] = pd.to_datetime(weather_df["Time"])

# Display dataset
print("Historical Weather Dataset")
display(weather_df.head(10))

print("\nDataset Shape:", weather_df.shape)

# Save to CSV
csv_file = "nairobi_historical_weather.csv"
weather_df.to_csv(csv_file, index=False)

print(f"\nCSV file '{csv_file}' created successfully.")

# Send data to MongoDB (replaces files.download)
from pymongo import MongoClient
from google.colab import userdata

# Connection string stored in Colab Secrets as MONGO_URI
MONGO_URI = userdata.get("MONGO_URI")

client = MongoClient(MONGO_URI)
db = client["weather_lab"]
collection = db["nairobi_hourly_weather"]

records = weather_df.to_dict("records")
collection.insert_many(records)

print(f"{len(records)} documents inserted into MongoDB.")
client.close()

Historical Weather Dataset


,Time,Temperature_C,Humidity_Percent,WindSpeed_mps
0,2026-09-22 00:00:00,16.2,89,7.0
1,2026-09-22 01:00:00,16.0,90,7.1
2,2026-09-22 02:00:00,15.8,91,7.1
3,2026-09-22 03:00:00,15.5,93,6.2
4,2026-09-22 04:00:00,15.5,93,5.8
5,2026-09-22 05:00:00,16.8,86,5.3
6,2026-09-22 06:00:00,18.9,73,6.1
7,2026-09-22 07:00:00,20.3,64,5.7
8,2026-09-22 08:00:00,22.7,54,8.9
9,2026-09-22 09:00:00,23.5,49,7.9



Dataset Shape: (720, 4)

CSV file 'nairobi_historical_weather.csv' created successfully.
720 documents inserted into MongoDB.
